In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
from tqdm import tqdm
import re
import locale
import urllib3
import emoji

# Suppress the InsecureRequestWarning specifically
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
def get_html(url = 'https://sindipetro-rs.org.br/todas-as-noticias/'):
    payload = {}
    headers = {}

    response = requests.request("GET", url, headers=headers, data=payload, verify=False)
    html_content = response.text

    return html_content

In [5]:
def get_links_and_dates(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    articles = soup.find_all('article')

    locale.setlocale(locale.LC_TIME, "pt_BR.utf8") 

    news_links = []

    for article in articles:
        a = article.find('a')
        link = a.get('href')
        span = article.find('span')
        date = span.text.strip()
        try:
            date = datetime.strptime(date, "%d de %B de %Y")
                
            link_date = [link, date]
            news_links.append(link_date)
        except ValueError as e:
            continue


    return news_links

In [13]:
def get_validated_links(news_links, min_date = datetime(2025,6,1)):
    validated_links = []
    next_page = True
    for link, date in news_links:
        if date < min_date:
            next_page = False
            break
        else:
            validated_links.append([link, date])

    return validated_links, next_page

In [12]:
def get_next_page(fnp_url = 'https://sindipetro-rs.org.br/todas-as-noticias/', next_page_number = 1):
    validated_news_links = []
    url = fnp_url + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(fnp_url, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    return validated_news_links

In [16]:
def sanitize_paragraphs(paragraphs:list, min_paragraph_len = 500, concat_trigger_size = 1500):
    '''
    sanitiza os parágrafos, realizando replace de partes de strings e concatenando parágrafos
    
    paragraphs: lista de parágrafos
    min_paragraph_len: define quais parágrafos devem ser concatenados
    concat_trigger_size: caso o tamanho da concatenação de textos esteja acima desse valor, 
                            escreve como parágrafo e reseta a variável que armazena as 
                            strings a serem concatenadas
    '''
    def append_paragraphs(current_new_paragraph):
        new_paragraphs.append(current_new_paragraph.strip())
    
    paragraphs = [emoji.demojize(paragraph) for paragraph in [paragraph\
                                                              .replace('\xa0',' ')\
                                                              .replace('\n',' ')\
                                                              .replace('\t',' ')\
                                                              .replace('[email-protected]', '')\
                                                              .strip() \
                                                              for paragraph in paragraphs] \
                  if len(paragraph)>0] #removendo emoji, tratanto texto e realizando strip para então construir lista de parágrafos que tenham len > 0
    paragraphs_mask = [len(paragraph) < min_paragraph_len for paragraph in paragraphs] # true são os abaixo do min_paragraph_len, precisarao ser tratados
    if any(paragraphs_mask): #caso algum elemento precise ser tratado, trigga processo
        new_paragraphs = []
        current_new_paragraph = ''
        for i in range(len(paragraphs)):
            if len(current_new_paragraph) >= concat_trigger_size:
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
            if paragraphs_mask[i]: #caso seja menor que o min_paragraph_len
                current_new_paragraph = current_new_paragraph + ' ' + paragraphs[i]
                if i == len(paragraphs)-1: #caso seja o ultimo elemento
                    append_paragraphs(current_new_paragraph)
            else:
                current_new_paragraph = current_new_paragraph + '' + paragraphs[i]
                append_paragraphs(current_new_paragraph)
                current_new_paragraph = ''
                    
    return new_paragraphs

In [17]:
def get_content_news(url):
    html_content = get_html(url)
    soup = BeautifulSoup(html_content, 'html.parser')
    
    title = soup.find('h1').text
    paragraphs = soup.find_all('p')

    paragraphs_text = [p.get_text(strip=True) for p in paragraphs]
    paragraphs = sanitize_paragraphs(paragraphs_text)

    return title, paragraphs

In [18]:
def main():
    next_page_number = 1
    validated_news_links = []
    url_default = 'https://sindipetro-rs.org.br/todas-as-noticias/'
    url = url_default + str(next_page_number) + '/'
    html_content = get_html(url)
    news_links = get_links_and_dates(html_content)
    validated_links, next_page = get_validated_links(news_links)

    for validated_link in validated_links:
        validated_news_links.append(validated_link)

    if next_page:
        validated_links = get_next_page(url_default, next_page_number + 1)
        
        for validated_link in validated_links:
            validated_news_links.append(validated_link)

    result = []
    for url, date in tqdm(validated_news_links):
        title, paragraphs = get_content_news(url)
        num_paragraph = 1
        for paragraph in paragraphs:
            result.append(
                {
                    'sindicato': 'RS',
                    'url' : url,
                    'titulo' : title,
                    'data': date,
                    'paragrafo' : paragraph,
                    'num_paragrafo' : num_paragraph
                }
            )
            num_paragraph += 1

    return result

In [19]:
result = main()
print(len(result))
result

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [01:00<00:00,  1.67s/it]

126


[{'sindicato': 'RS',
  'url': 'https://sindipetro-rs.org.br/fup-e-sindicatos-participam-de-oficina-de-planejamento-da-campanha-reivindicatoria/',
  'titulo': 'FUP e sindicatos participam de oficina de planejamento da campanha reivindicatória',
  'data': datetime.datetime(2025, 8, 19, 0, 0),
  'paragrafo': 'Dirigentes da FUP e de seus sindicatos participam da Oficina de Planejamento de Campanha dos Trabalhadores e Trabalhadoras do Sistema Petrobrás, com o objetivo de construir um plano de ação para conquistar as principais reivindicações que foram deliberadas na19ª Plenafup.A atividade, conduzida pelo Dieese, teve início nesta terça, 19, e prossegue até quinta-feira, 21, na sede da Associação de Aposentados e Funcionários do Banco do Brasil, em Xerém, região metropolitana do Rio de Janeiro.A atividade, conduzida pelo Dieese, teve início nesta terça, 19, e prossegue até quinta-feira, 21, na sede da Associação de Aposentados e Funcionários do Banco do Brasil, em Xerém, região metropolitan